In [1]:
from dataset import get_train_transforms,get_val_transforms,ImageDataset
from model import ResNet18,ImageSSL,LinearProbe
from losses import VICRegLoss
from lars import LARS
from scheduler import WarmupCosineScheduler

import torch 
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
import torch.nn.functional as F 

# Data 

In [2]:
train_data =  CIFAR10(root="/tmp/cifar10",train=True,download=True)

train_ds = ImageDataset(
    dataset= train_data, 
    transform= get_train_transforms(),
    num_crops= 2 
)

val_ds =  CIFAR10(root="/tmp/cifar10",train=False,download=True,transform=get_val_transforms())


train_loader = DataLoader(train_ds,batch_size=32,shuffle=False)
val_loader   = DataLoader(val_ds,    batch_size=32,shuffle=False)

In [3]:
print(len(train_loader))
print(len(val_loader))

1563
313


In [4]:
for batch in train_loader: 
    print(len(batch))
    views,label = batch 
    view0,view1 = views
    print(view0.shape,view1.shape)
    print(label.shape)
    print(label)
    break 

2
torch.Size([32, 3, 32, 32]) torch.Size([32, 3, 32, 32])
torch.Size([32])
tensor([6, 9, 9, 4, 1, 1, 2, 7, 8, 3, 4, 7, 7, 2, 9, 9, 9, 3, 2, 6, 4, 3, 6, 6,
        2, 6, 3, 5, 4, 0, 0, 9])


In [5]:
for batch in val_loader: 
    print(len(batch ))
    x,y = batch 
    print(x.shape)
    print(y.shape)
    break 

2
torch.Size([32, 3, 32, 32])
torch.Size([32])


# Model 

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

backbone = ResNet18()

features_dim = backbone.features_dim

model = ImageSSL(
    backbone= backbone,
    features_dim = features_dim,    # 512
    proj_hidden_dim=2048,   # from cfgs/default.yaml
    proj_output_dim=2048    # from cfgs/default.yaml

)
model = model.to(device)

linear_probe = LinearProbe(feature_dim=features_dim, num_classes=10).to(device) 


# Loss,Optimizer,Scheduler

In [7]:
loss_fn = VICRegLoss(std_coeff= 1.0, cov_coeff= 80.0)
optimizer = LARS(
    [
        {"params":model.parameters(),"lr":0.3},
        {"params":linear_probe.parameters(),"lr":0.1}
    ],
    weight_decay=1.0e-4,
    eta = 0.02,
    clip_lr=True,
    exclude_bias_n_norm=True,
    momentum=0.9
)

scheduler = WarmupCosineScheduler(
    optimizer,
    warmup_epochs= 10,
    max_epochs= 300,
    base_lr=0.3,
    min_lr=0.0,
    warmup_start_lr=3.0e-5
)

# Train 

In [8]:

for epoch in range(100):

    model.train()
    linear_probe.train()
    epoch_loss = 0 

    for views,labels in train_loader: 
        #============================== forward pass 
        view0,view1 = views
        features,z1 = model(view0)
        _,       z2 = model(view1)
        print(features.shape,z1.shape)
        #==============================
        ssl_loss = loss_fn(z1,z2)["loss"]
        probe_loss = F.cross_entropy(linear_probe(features.detach()),labels)    # use y_hat
        #==============================
        total_loss = ssl_loss + probe_loss 
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        epoch_loss += ssl_loss.item()
        print(epoch_loss)
        break 

    scheduler.step(epoch)
    avg_loss = epoch_loss / len(train_loader)
    # print(f'Epoch {epoch}: loss = {avg_loss:.4f}')

torch.Size([32, 512]) torch.Size([32, 2048])
1.9060713052749634
torch.Size([32, 512]) torch.Size([32, 2048])
1.679402232170105
torch.Size([32, 512]) torch.Size([32, 2048])
1.6943061351776123
torch.Size([32, 512]) torch.Size([32, 2048])
1.6232976913452148
torch.Size([32, 512]) torch.Size([32, 2048])
1.6250768899917603
torch.Size([32, 512]) torch.Size([32, 2048])
1.618448257446289
torch.Size([32, 512]) torch.Size([32, 2048])
1.6141936779022217
torch.Size([32, 512]) torch.Size([32, 2048])
1.6131473779678345
torch.Size([32, 512]) torch.Size([32, 2048])
1.59708833694458
torch.Size([32, 512]) torch.Size([32, 2048])
1.5888093709945679
torch.Size([32, 512]) torch.Size([32, 2048])
1.597856044769287
torch.Size([32, 512]) torch.Size([32, 2048])
1.6119067668914795
torch.Size([32, 512]) torch.Size([32, 2048])
1.5983277559280396
torch.Size([32, 512]) torch.Size([32, 2048])
1.5969314575195312
torch.Size([32, 512]) torch.Size([32, 2048])
1.5995386838912964
torch.Size([32, 512]) torch.Size([32, 2048])


# Evaluation 

In [9]:
from torch.amp import autocast 

def evaluate_linear_porbe(model,linear_probe,val_loader,device,use_amp=True): 
    model.eval()
    linear_probe.eval()

    total_loss = 0 
    correct = 0 
    total = 0 

    with torch.no_grad(): 
        for data,target in val_loader:
            data = data.to(device,non_blocking=True)
            target = target.to(device,non_blocking = True)

            with autocast("cuda",enabled=use_amp):
                features, _ = model(data)

            outputs = linear_probe(features.float())
            # print(outputs)
            loss = F.cross_entropy(outputs,target)

            total_loss += loss.item()
            _,predicted = outputs.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            break 

    accuracy = 100 * correct / total
    avg_loss = total_loss /len(val_loader)

    return accuracy,avg_loss


In [10]:
evaluate_linear_porbe(model,linear_probe,val_loader,device=device,use_amp=False)

(15.625, 0.012497972756529007)